# Research Grade Hydroponics Simulator

This notebook provides a comprehensive environment for simulating and managing a hydroponic crop system. It integrates theoretical physics, deep learning, and real-time synchronization with cloud services.

**Key Features:**
*   **Physics-Informed Core**: Combines first-principles differential equations with a Physics-Informed Neural Network (PINN) for "biological chaos" modeling.
*   **Reinforcement Learning**: Built-in Gymnasium environment for training optimal control agents using Proximal Policy Optimization (PPO).
*   **Azure Digital Twins**: Real-time telemetry synchronization with Azure-hosted Digital Twins.
*   **Research API**: Exposes a FastAPI server with ngrok tunneling for remote telemetry and control.

## Setup & Dependencies

In [ ]:
!pip install torch azure-digitaltwins-core azure-identity Pillow numpy fastapi uvicorn nest_asyncio google-genai gymnasium stable-baselines3 pyngrok -q
!pip install diffusers transformers accelerate torchvision -q

In [ ]:
import base64
import time
import os
import json
import threading
import uvicorn
import nest_asyncio
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from io import BytesIO
from collections import deque
from dataclasses import dataclass, asdict
from fastapi import FastAPI
from pydantic import BaseModel
from PIL import Image
from stable_baselines3 import PPO
from azure.identity import DeviceCodeCredential
from azure.digitaltwins.core import DigitalTwinsClient
from pyngrok import ngrok

nest_asyncio.apply()

# Configuration
MODEL_PATH = "models/PPO/lettuce_brain_v1.zip"
HISTORY_LEN = 20
ADT_URL = "simulator.api.krc.digitaltwins.azure.net"
TWIN_ID = "HydrophonicTank"

## 1. Physics-Informed Architecture

Defining the `ResidualPhysicsNet` which captures non-linear biological effects that standard textbook equations often miss.

In [ ]:
class ResidualPhysicsNet(nn.Module):
    def __init__(self, state_dim=7, action_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, state_dim)
        )
        with torch.no_grad():
            self.net[-1].weight.mul_(0.01)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        return self.net(x)

## 2. Gymnasium Environment

Standardized RL interface for training control agents.

In [ ]:
class HydroponicsEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.high_obs = np.array([14.0, 5.0, 40.0, 50.0, 100.0, 5.0, 5000.0], dtype=np.float32)
        self.low_obs = np.array([0.0, 0.0, 5.0, 0.0, 0.0, 0.0, 0.0], dtype=np.float32)
        self.observation_space = spaces.Box(low=self.low_obs, high=self.high_obs, dtype=np.float32)
        self.action_space = spaces.Box(low=0.0, high=1.0, shape=(4,), dtype=np.float32)
        
        self.state = None
        self.residual_model = ResidualPhysicsNet(7, 4)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.array([
            6.0 + np.random.uniform(-0.2, 0.2), # pH
            1.5 + np.random.uniform(-0.1, 0.1), # EC
            20.0, 24.0, 60.0, 1.0, 10.0
        ], dtype=np.float32)
        return self.state, {}

    def _physics_prior(self, state, action):
        ph, ec, w_temp, a_temp, hum, vpd, biomass = state
        acid, base, nutes, fan = action
        d_ph = (base * 0.5) - (acid * 0.5) + (0.01 * biomass / 1000)
        uptake = 0.05 * biomass * vpd
        d_ec = (nutes * 0.2) - (uptake / 100.0)
        stress = np.abs(vpd - 1.0)
        growth_rate = 0.1 * (1.0 - min(stress, 1.0))
        d_biomass = biomass * growth_rate
        d_temp = -1.0 * fan + 0.1 
        d_hum = -5.0 * fan + 2.0
        new_vpd = 0.61 * np.exp((17.27 * a_temp)/(a_temp+237.3)) * (1 - hum/100)
        d_vpd = new_vpd - vpd
        return np.array([d_ph, d_ec, 0, d_temp, d_hum, d_vpd, d_biomass], dtype=np.float32)

    def step(self, action):
        action = np.array(action, dtype=np.float32)
        d_physics = self._physics_prior(self.state, action)
        with torch.no_grad():
            st_t = torch.tensor(self.state, dtype=torch.float32)
            at_t = torch.tensor(action, dtype=torch.float32)
            d_residual = self.residual_model(st_t, at_t).numpy()
            
        self.state += d_physics + (d_residual * 0.1)
        self.state = np.clip(self.state, self.low_obs, self.high_obs)
        ph, ec, _, _, _, _, biomass = self.state
        reward = -abs(ph - 6.0) * 10.0 - abs(ec - 1.5) * 5.0 + biomass * 0.1
        terminated = bool(ph < 4.0 or ph > 8.0)
        return self.state, float(reward), terminated, False, {}

## 3. Agent Training (PPO)

Train the agent using Stable Baselines3. This simulates the optimization of resource usage for growth.

In [ ]:
if __name__ == "__main__":
    env = HydroponicsEnv()
    os.makedirs("models/PPO", exist_ok=True)
    os.makedirs("logs", exist_ok=True)

    model = PPO("MlpPolicy", env, verbose=1, tensorboard_log="logs")
    print("🤖 Training Agent...")
    model.learn(total_timesteps=10000)
    
    model.save(MODEL_PATH)
    print(f"💾 Model saved to: {MODEL_PATH}")

## 4. Cloud Integration (Azure Digital Twins)

Logic for synchronizing local simulator state with a cloud-hosted Digital Twin.

In [ ]:
try:
    credential = DeviceCodeCredential()
    client = DigitalTwinsClient(ADT_URL, credential)
    
    def sync_to_azure(state):
        ph, ec, water_temp, air_temp, humidity, vpd, biomass = state
        payload = {
            "ph": float(ph),
            "ec": float(ec),
            "water_temp": float(water_temp),
            "air_temp": float(air_temp),
            "humidity": float(humidity),
            "vpd": float(vpd),
            "biomass_g": float(biomass)
        }
        client.publish_telemetry(TWIN_ID, payload)

    def initialize_twin_state():
        initial_patch = [
            {"op": "add", "path": "/ph", "value": 6.0},
            {"op": "add", "path": "/ec", "value": 1.5},
            {"op": "add", "path": "/biomass_g", "value": 10.0}
        ]
        client.update_digital_twin(TWIN_ID, initial_patch)
        print("✅ Twin initialized!")
except NameError:
    print("⚠️ Azure packages or identity missing. Skipping initialization.")
except Exception as e:
    print(f"⚠️ Azure error: {e}")

## 5. Research Simulator API & Ngrok Gateway

The simulation engine and FastAPI server provide a programmatic interface for remote monitoring and action.

In [ ]:
@dataclass
class FarmStateData:
    ph: float; ec: float; water_temp: float; air_temp: float
    humidity: float; vpd: float; biomass_g: float; tank_volume_l: float

class FarmAction(BaseModel):
    acid_dosage_ml: float = 0.0
    base_dosage_ml: float = 0.0
    nutrient_dosage_ml: float = 0.0
    fan_speed_pct: float = 0.0
    debug_force_ph: float | None = None

class DigitalTwin:
    def __init__(self):
        self.state = np.array([6.0, 1.5, 20.0, 24.0, 60.0, 1.0, 10.0], dtype=np.float32)
        self.plant_health = 100.0
        self.residual_model = ResidualPhysicsNet(7, 4)
        self.history = {k: deque([v]*5, maxlen=HISTORY_LEN) for k,v in zip(["ph", "ec", "water_temp", "air_temp", "humidity", "vpd"], self.state[:6])}

    def step(self, action: FarmAction = None):
        if action is None: action = FarmAction()
        u = np.array([action.acid_dosage_ml/10.0, action.base_dosage_ml/10.0, action.nutrient_dosage_ml/20.0, action.fan_speed_pct/100.0], dtype=np.float32)
        
        ph, ec, wt, at, hum, vpd, bio = self.state
        d_physics = np.array([(u[1]-u[0])*0.5 + 0.001*bio, u[2]*0.2 - (0.02*bio*vpd)/100.0, 0, 0.1 - u[3]*1.5, 1.0 - u[3]*5.0, 0, 0.1*bio*(1.0-abs(vpd-1.0))], dtype=np.float32)
        
        with torch.no_grad():
            nn_delta = self.residual_model(torch.tensor(self.state), torch.tensor(u)).numpy()
        
        self.state += d_physics + (nn_delta * 0.05)
        self.state[0] = action.debug_force_ph if action.debug_force_ph else self.state[0]
        self.state[3:5] = np.clip(self.state[3:5], [0, 0], [50, 100])
        
        for i, k in enumerate(self.history.keys()): self.history[k].append(float(self.state[i]))
        return Image.new('RGB', (512, 512), (50, 50, 50))

app = FastAPI()
sim = DigitalTwin()

@app.get("/simulation/state")
async def get_state():
    img = sim.step()
    buf = BytesIO(); img.save(buf, format="PNG")
    return {"sensor_window": {k: list(v) for k,v in sim.history.items()}, "metadata": {"health": round(float(sim.plant_health), 1), "biomass": round(float(sim.state[6]), 2)}, "image": base64.b64encode(buf.getvalue()).decode("utf-8")}

@app.post("/simulation/action")
async def handle_action(action: FarmAction):
    sim.step(action)
    try: sync_to_azure(sim.state)
    except: pass
    return {"status": "success", "ph": float(sim.state[0])}

def run_api():
    print("🚀 Research Simulator API Online (Port 3001)")
    uvicorn.run(app, host="0.0.0.0", port=3001, log_level="error")

threading.Thread(target=run_api, daemon=True).start()

# Exposed via Ngrok
try:
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(3001).public_url
    print(f"🌍 Tunnel Ready: {public_url}")
except Exception as e: print(f"⚠️ Ngrok failed: {e}")